In [1]:
# 加载环境变量
from dotenv import find_dotenv, load_dotenv
from langchain_core.messages import AIMessage

load_dotenv(find_dotenv(".env", usecwd=True), override=True)


True

# 定义工具


In [2]:
from langchain_tavily import TavilySearch

# web搜索工具，使用tavily作为web搜索工具
web_search = TavilySearch(
    max_results=5,
    topic="general"
)

# 定义模型
我们定义两个模型，一个多模态模型，一个聊天模型


In [3]:
from langchain.chat_models import init_chat_model
import os

# 多模态模型
multimodal_model = init_chat_model(
    model=os.getenv("MODEL_NAME"),  # 模型名称，从 .env 中读取
    model_provider="openai",
    base_url=os.getenv("BASE_URL"),
    api_key=os.getenv("SILICONFLOW_API_KEY")
)


# 添加记忆管理

这里我们采用sqllite

In [4]:
from langgraph.checkpoint.sqlite import SqliteSaver
import sqlite3

# 初始化checkpointer
checkpointer = SqliteSaver(sqlite3.connect("resources/personal_chief.db", check_same_thread=False))
# 自动建表
checkpointer.setup()

# 定义智能体

In [5]:
from langchain.agents import create_agent

system_prompt = """
你是一名私人厨师。收到用户提供的食材照片或清单后，请按以下流程操作：
1.识别和评估食材：若用户提供照片，首先辨识所有可见食材。基于食材的外观状态，评估其新鲜度与可用量，整理出一份“当前可用食材清单”。
2.智能食谱检索：优先调用 web_search 工具，以“可用食材清单”为核心关键词，查找可行菜谱。
3.多维度评估与排序：从营养价值和制作难度两个维度对检索到的候选食谱进行量化打分，并根据得分排序，制作简单且营养丰富的排名靠前。
4.结构化方案输出：把排序后的食谱整理为一份结构清晰的建议报告，要包含食谱信息、得分、推荐理由，帮助用户快速做出决策。

请严格按照流程，优先调用 web_search 工具搜索食谱，再搜索不到的情况下才能自己发挥。
"""

agent = create_agent(
    model=multimodal_model,
    tools=[web_search],
    system_prompt=system_prompt,
    checkpointer=checkpointer
)

# 测试


In [6]:
from langchain.messages import HumanMessage

multimodal_message = HumanMessage(
    content=[
        {"type": "text", "text": "帮我看看这些食材能做些什么？"},
        {"type": "image",
         "url": "https://personuse.oss-cn-beijing.aliyuncs.com/1778644466545.png"}
    ])

config = {"configurable": {"thread_id": "1"}}

In [7]:
response = agent.invoke({"messages": [multimodal_message]}, config)

BadRequestError: Error code: 400 - {'code': 20040, 'message': 'Unable to process the image', 'data': None}

In [ ]:
# 友好打印
for message in response['messages']:
    message.pretty_print()

In [ ]:
response = agent.invoke(
    {"messages": [HumanMessage(content="我喜欢第3道菜，可以说的更详细点吗？")]},
    config
)

In [ ]:
# 友好打印
response['messages'][-1].pretty_print()

# 流式调用

有工具调用的agent消息类型比较多，要正确的流式输出比较麻烦，这里给出一个示例


In [ ]:
from langchain.messages import AIMessageChunk

for chunk, metadata in agent.stream(
            {"messages": [multimodal_message]},
            config,
            stream_mode="messages"
        ):
            if isinstance(chunk, AIMessageChunk) and chunk.content:
                print(chunk.content, end="", flush=True)

# 获取会话历史

既然会话记忆交给了checkpointer管理，我们自然可以从中获取会话历史


In [ ]:
checkpointer.get(config)

In [ ]:
for m in checkpointer.get(config)['channel_values']['messages']:
    print(type(m))
    print(m)

定义一个便捷的获取会话历史的函数：


In [ ]:
def get_messages(thread_id: str) -> list[dict[str, str]]:
    """获取会话历史"""

    # 根据 thread_id 查询 checkpoint
    cp = checkpointer.get({"configurable": {"thread_id": thread_id}})

    # 如果不存在，返回空列表
    if not cp:
        return []

    # 安全获取 messages
    channel_values = cp.get("channel_values")
    if not channel_values:
        return []

    messages = channel_values.get("messages", [])
    if not messages:
        return []

    # 转换消息格式
    result = []
    for msg in messages:
        if not msg.content:
            continue

        if isinstance(msg, HumanMessage):
            result.append({"role": "user", "content": msg.content})
        elif isinstance(msg, AIMessage):
            result.append({"role": "assistant", "content": msg.content})

    return result

In [ ]:
get_messages("1")

# 清空会话历史


In [ ]:
checkpointer.delete_thread("1")